# Deployment Smoke Test

This notebook checks that the saved model artifact can be loaded and used for a new prediction.

The pickle artifact contains the complete preprocessing pipeline, the classifier, the target-label encoder, and the feature names. The application can therefore use the same training-time transformations without recreating them by hand.

In [ ]:
from pathlib import Path

import pickle
import pandas as pd


def find_project_root() -> Path:
    current_folder = Path.cwd().resolve()
    for folder in [current_folder, *current_folder.parents]:
        if (folder / "src" / "data" / "companies.csv").exists():
            return folder
    raise FileNotFoundError("Could not find src/data/companies.csv")


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "src" / "data" / "companies.csv"
MODEL_PATH = PROJECT_ROOT / "src" / "models" / "best_model.pkl"
TARGET_COLUMN = "status"
columns_to_drop = [
    TARGET_COLUMN, "id", "Unnamed: 0.1", "entity_type", "entity_id", "parent_id",
    "name", "normalized_name", "permalink", "domain", "homepage_url",
    "twitter_username", "logo_url", "logo_width", "logo_height", "short_description",
    "description", "overview", "tag_list", "created_by", "created_at", "updated_at",
    "first_investment_at", "last_investment_at", "first_funding_at", "last_funding_at",
    "first_milestone_at", "last_milestone_at", "closed_at", "ROI",
]

with MODEL_PATH.open("rb") as model_file:
    model_artifact = pickle.load(model_file)
model = model_artifact["pipeline"]
target_encoder = model_artifact["target_encoder"]
data = pd.read_csv(DATA_PATH)
feature_columns = model_artifact["feature_columns"]
print(f"Loaded {model_artifact['model_name']}: {MODEL_PATH}")
print(f"Expected feature count: {len(feature_columns)}")

In [ ]:
def predict_status(company_features: pd.DataFrame) -> str:
    """Return the readable startup status for one or more feature rows."""
    missing_columns = set(feature_columns) - set(company_features.columns)
    if missing_columns:
        raise ValueError(f"Missing required feature columns: {sorted(missing_columns)}")

    prediction_codes = model.predict(company_features[feature_columns]).astype(int)
    return target_encoder.inverse_transform(prediction_codes)[0]


example_company = data[feature_columns].iloc[[0]]
predicted_status = predict_status(example_company)
print(f"Predicted status for the example company: {predicted_status}")

In [ ]:
# This is the shape an API or batch job can provide to the prediction function.
request_data = example_company.to_dict(orient="records")
print(request_data[0])
print({"prediction": predicted_status})

## Production notes

- Keep `src/models/best_model.pkl` versioned with the code that created it.
- Validate incoming feature names before prediction.
- Never fit preprocessing during an API request.
- Monitor predictions and real outcomes after deployment.
- A future FastAPI endpoint can load the same pickle file and pass a DataFrame with the same feature columns.